# Framework: Core ML Pipeline

This notebook contains the reusable ML pipeline functions that can be used for any train/test split.

**Key Functions:**
- `train_and_evaluate()`: Main function that takes train/test matrices and returns performance metrics and selected features
- `generate_synthetic_features()`: Creates synthetic features from feature pairs
- `run_mdfs_feature_selection()`: Runs MDFS to identify feature pairs for synthetic features
- `find_best_k()`: Finds optimal number of features using internal validation
- `plot_roc_curve()`: Visualization utility

**Usage:** This framework is called by `main_code.ipynb` for each LOCO fold, but can also be used standalone for any train/test split.

In [13]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score
from sklearn.model_selection import train_test_split
import warnings
import subprocess
import os
import matplotlib.pyplot as plt

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

# --- MODEL AND FEATURE SELECTION PARAMETERS ---
TOP_K_VALUES = [50, 100, 150, 200, 250, 300, 350, 400, 450, 500]
MODEL_PARAMS = {
    'n_estimators': 1000, 'max_features': 'sqrt', 'min_samples_leaf': 1,
    'class_weight': 'balanced', 'random_state': 42, 'n_jobs': -1
}

SYNTHETIC_PREFIXES = ('LR_', 'GM_')

In [14]:
def generate_synthetic_features(X, feature_pairs, epsilon=1e-9):
    """
    Generates synthetic features from feature pairs.
    
    Parameters:
    -----------
    X : pd.DataFrame
        Feature matrix (samples x features)
    feature_pairs : list of tuples
        List of (feature1, feature2) pairs to generate synthetic features from
    epsilon : float
        Small value to avoid division by zero
    
    Returns:
    --------
    pd.DataFrame
        Synthetic features matrix with columns: LR_*, GM_*
    """
    synthetic_X = pd.DataFrame(index=X.index)
    for pair in feature_pairs:
        f1, f2 = sorted(pair)
        if f1 in X.columns and f2 in X.columns:
            # Log-ratio (division)
            synthetic_X[f"LR_{f1}__{f2}"] = np.log((X[f1] + epsilon) / (X[f2] + epsilon))
            # Geometric mean
            synthetic_X[f"GM_{f1}__{f2}"] = np.sqrt((X[f1] + epsilon) * (X[f2] + epsilon))
    return synthetic_X

In [15]:
def find_best_k(X_train, y_train, k_values, model_params):
    """
    Finds the best k (number of features) using an internal train-validation split.
    
    Parameters:
    -----------
    X_train : pd.DataFrame
        Training feature matrix
    y_train : pd.Series
        Training labels
    k_values : list
        List of k values to try
    model_params : dict
        Parameters for RandomForestClassifier
    
    Returns:
    --------
    int
        Best k value based on validation AUC
    """
    if X_train.shape[1] < min(k_values): 
        return X_train.shape[1]
    
    X_train_transformed = X_train.copy()
    
    X_train_internal, X_val_internal, y_train_internal, y_val_internal = train_test_split(
        X_train_transformed, y_train, test_size=0.25, random_state=42, stratify=y_train)

    ranking_model = RandomForestClassifier(**model_params).fit(X_train_internal, y_train_internal)
    sorted_features = pd.Series(ranking_model.feature_importances_, index=X_train_internal.columns).sort_values(ascending=False).index

    best_auc, best_k = -1, k_values[0]
    for k in k_values:
        k = min(k, len(sorted_features))
        top_k_features = sorted_features[:k]
        model = RandomForestClassifier(**model_params).fit(X_train_internal[top_k_features], y_train_internal)
        y_pred_proba = model.predict_proba(X_val_internal.reindex(columns=top_k_features, fill_value=0))[:, 1]
        try:
            auc = roc_auc_score(y_val_internal, y_pred_proba)
            if auc > best_auc:
                best_auc, best_k = auc, k
        except ValueError: 
            continue
    return best_k

In [ ]:
def run_mdfs_feature_selection(X_train, y_train, temp_prefix='temp', return_ig_info=False):
    """
    Runs MDFS to identify feature pairs for synthetic feature generation.
    
    Parameters:
    -----------
    X_train : pd.DataFrame
        Training feature matrix
    y_train : pd.Series
        Training labels
    temp_prefix : str
        Prefix for temporary files
    return_ig_info : bool
        If True, returns dict with pairs, IG values, and individual feature IGs
    
    Returns:
    --------
    If return_ig_info=False:
        list of tuples: List of (feature1, feature2) pairs
    If return_ig_info=True:
        dict with keys:
            - 'pairs': list of (feature1, feature2) tuples
            - 'pair_ig': dict mapping (base, contributing) -> total_IG
            - 'individual_ig': dict mapping feature -> IG value (1D)
    """
    train_X_file = f"{temp_prefix}_X.tsv"
    train_y_file = f"{temp_prefix}_y.tsv"
    pairs_file = f"{temp_prefix}_pairs.txt"
    mdfs_file = f"{temp_prefix}_mdfs.txt"
    ig_1d_file = f"{temp_prefix}_1d_ig.txt"
    
    feature_pairs = []
    pair_ig_dict = {}
    individual_ig_dict = {}
    
    try:
        print("  Running MDFS to identify feature pairs for synthetic features...")
        X_train.to_csv(train_X_file, sep="\t")
        y_train.to_csv(train_y_file, sep="\t", header=True)
        
        # Check if run_mdfs.R exists in current directory or parent
        rscript_path = "run_mdfs.R"
        if not os.path.exists(rscript_path):
            rscript_path = "../run_mdfs.R"
        
        subprocess.run(
            ["Rscript", rscript_path, train_X_file, train_y_file, pairs_file, mdfs_file, ig_1d_file],
            check=True, capture_output=True, text=True
        )
        
        # Read 1D IG values (new format: Feature, IG, p_value, adjusted_p_value)
        individual_ig_dict = {}
        if os.path.exists(ig_1d_file):
            try:
                ig_1d_df = pd.read_csv(ig_1d_file, sep="\t")
                for _, row in ig_1d_df.iterrows():
                    feat = str(row['Feature']).replace(' ', '_')
                    individual_ig_dict[feat] = float(row['IG'])
            except Exception as e:
                print(f"  Warning: Could not read 1D IG from MDFS output: {e}")
        
        # Read 2D tuples (new format: base, contributing, base_IG_1D,
        # contributing_IG_1D, IG_2D_added, total_IG)
        # These are already filtered to statistically significant 2D features
        mdfs_tuples = pd.read_csv(pairs_file, sep="\t")
        
        if not mdfs_tuples.empty and 'base' in mdfs_tuples.columns:
            # Extract unique pairs (each pair appears twice — once per direction)
            seen_pairs = set()
            for _, r in mdfs_tuples.iterrows():
                base = str(r['base']).replace(' ', '_')
                contrib = str(r['contributing']).replace(' ', '_')
                pair_key = tuple(sorted([base, contrib]))
                if pair_key not in seen_pairs:
                    seen_pairs.add(pair_key)
                    feature_pairs.append(pair_key)
                # Store IG for both directions (base, contributing) -> total_IG
                pair_ig_dict[(base, contrib)] = float(r['total_IG'])
        
        print(f"  Identified {len(feature_pairs)} synthetic feature pairs.")
    except Exception as e:
        error_output = e.stderr if hasattr(e, 'stderr') else str(e)
        print(f"  MDFS Error: {error_output}. No synthetic features will be used.")
    finally:
        for f in [train_X_file, train_y_file, pairs_file, mdfs_file, ig_1d_file]:
            if os.path.exists(f):
                os.remove(f)
    
    if return_ig_info:
        return {
            'pairs': feature_pairs,
            'pair_ig': pair_ig_dict,
            'individual_ig': individual_ig_dict
        }
    else:
        return feature_pairs

In [17]:
def train_and_evaluate(X_train, y_train, X_test, y_test, 
                       k_values=None, model_params=None, 
                       return_model=False, return_predictions=False):
    """
    Core ML pipeline: trains a Random Forest model and evaluates on test set.
    
    This function:
    1. Finds optimal number of features (k) using internal validation
    2. Ranks features by importance
    3. Trains final model on top k features
    4. Evaluates on test set
    5. Returns performance metrics and selected features
    
    Parameters:
    -----------
    X_train : pd.DataFrame
        Training feature matrix (samples x features)
    y_train : pd.Series
        Training labels (0 = CRC, 1 = Control)
    X_test : pd.DataFrame
        Test feature matrix (samples x features)
    y_test : pd.Series
        Test labels (0 = CRC, 1 = Control)
    k_values : list, optional
        List of k values to try for feature selection (default: TOP_K_VALUES)
    model_params : dict, optional
        Parameters for RandomForestClassifier (default: MODEL_PARAMS)
    return_model : bool
        Whether to return the trained model (default: False)
    return_predictions : bool
        Whether to return predictions (default: False)
    
    Returns:
    --------
    dict
        Dictionary containing:
        - 'auc': AUC score
        - 'accuracy': Accuracy score
        - 'best_k': Number of features selected
        - 'selected_features': List of selected feature names
        - 'model': Trained model (if return_model=True)
        - 'y_pred_proba': Predicted probabilities (if return_predictions=True)
        - 'y_pred': Predicted classes (if return_predictions=True)
    """
    if k_values is None:
        k_values = TOP_K_VALUES
    if model_params is None:
        model_params = MODEL_PARAMS
    
    # Check if we have enough features
    if X_train.empty or X_train.shape[1] < 10:
        return {
            'auc': np.nan,
            'accuracy': np.nan,
            'best_k': 0,
            'selected_features': []
        }
    
    # Find best k using internal validation
    best_k = find_best_k(X_train, y_train, k_values, model_params)
    
    # Train ranking model to get feature importances
    ranking_model = RandomForestClassifier(**model_params).fit(X_train, y_train)
    sorted_features = pd.Series(ranking_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
    top_features = sorted_features.nlargest(best_k).index.tolist()
    
    # Train final model on selected features
    final_model = RandomForestClassifier(**model_params).fit(X_train[top_features], y_train)
    
    # Prepare test set (align features, fill missing with 0)
    X_test_aligned = X_test.reindex(columns=top_features, fill_value=0)
    
    # Predict
    y_pred_proba = final_model.predict_proba(X_test_aligned)[:, 1]
    y_pred = final_model.predict(X_test_aligned)
    
    # Calculate metrics
    auc = roc_auc_score(y_test, y_pred_proba)
    accuracy = accuracy_score(y_test, y_pred)
    
    # Build result dictionary
    result = {
        'auc': auc,
        'accuracy': accuracy,
        'best_k': best_k,
        'selected_features': top_features
    }
    
    if return_model:
        result['model'] = final_model
    
    if return_predictions:
        result['y_pred_proba'] = y_pred_proba
        result['y_pred'] = y_pred
    
    return result

In [18]:
def plot_roc_curve(y_true, y_pred_proba, title="ROC Curve"):
    """
    Plots ROC curve and returns the figure.
    
    Parameters:
    -----------
    y_true : array-like
        True labels
    y_pred_proba : array-like
        Predicted probabilities
    title : str
        Plot title
    
    Returns:
    --------
    matplotlib.figure.Figure
        The figure object
    """
    fpr, tpr, _ = roc_curve(y_true, y_pred_proba)
    auc_score = roc_auc_score(y_true, y_pred_proba)
    
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {auc_score:.4f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random classifier')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.title(title, fontsize=14)
    plt.legend(loc="lower right", fontsize=10)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    return plt.gcf()